# COMP0228 — Optical communication coursework

**This is the main notebook. Run it from top to bottom.**

We send bits using a flashing LCD, record them with a phone, recover the message and investigate how coding and signal processing change reliability and useful transmission rate.

1. Understand the channel and packet format.
2. Inspect the physical evidence and reproduce the baseline.
3. Evaluate the receiver-processing extension.
4. Compare codes under controlled synthetic noise.
5. Evaluate the interleaving extension and draw conclusions.

**Run All performs no hardware transmission.** It re-evaluates the physical feature caches, verifies all saved simulation records and replays a deterministic subset of simulations. A full simulation rerun is optional.

The physical evidence contains **30 packets in 14 videos**. The simulation contains **480,000 decodes**, which are not additional physical observations. Keep those sources of evidence distinct.

**All analysis data is included in Git.** Silent videos are optional release downloads; the video data and timing are unchanged. Normal Run All needs no video download; see the README and `code/tools/get_recordings.py`.


## 1. Set up the workspace

In [ ]:
from pathlib import Path
import sys, json, hashlib
import numpy as np
import pandas as pd
from IPython.display import display, Image

PROJECT = Path.cwd().resolve()
if PROJECT.name == 'code' and (PROJECT.parent / 'Coursework.ipynb').is_file():
    PROJECT = PROJECT.parent
CODE_ROOT = PROJECT / 'code'
DATA_ROOT = PROJECT / 'data'
assert (CODE_ROOT / 'optical_link').is_dir(), 'Open this notebook from the optical_link project folder.'
if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))
ROOT = CODE_ROOT
SIM_STUDY = DATA_ROOT / 'analysis/coding_experiments_2026-10-24'
from optical_link.protocol import make_packet, CODECS, GUARD_BITS, PREAMBLE, HEADER_BITS
from optical_link.coding_experiments import digest
print('Project:', PROJECT)
print('Data:', DATA_ROOT)


## 2. Channel, encoding and transmission cost

```text
Message → CRC + optional coding → ESP32/LCD flashes → phone video
Message ← CRC check + decoding ← sampled bits     ← brightness trace
```

White represents one; black represents zero. Each bit lasts 200 ms. A public preamble establishes timing and brightness levels; a CRC-protected header supplies the codec, payload length and sequence ID. The body contains the message and its CRC, optionally protected by an error-correcting code.

We compare uncoded transmission, repetition ×3 and ×5, Hamming (7,4), and Reed–Solomon with four parity bytes. The RS implementation is a credited third-party library; our work integrates and evaluates it. CRC detects errors; it does not correct them.

The display firmware is in [code/firmware/optical_tx](code/firmware/optical_tx/). Packet encoding is in [protocol.py](code/optical_link/protocol.py); camera decoding is in [receiver.py](code/optical_link/receiver.py).


In [ ]:
MESSAGE = 'MANUNITED'
costs = []
for codec in CODECS:
    packet = make_packet(MESSAGE, codec, sequence=23)
    seconds = len(packet.bits) * 0.2
    costs.append(dict(codec=codec, message_bits=len(packet.payload)*8,
                      protected_body_bits=len(packet.body_bits), full_packet_bits=len(packet.bits),
                      airtime_seconds=seconds, maximum_useful_bps=len(packet.payload)*8/seconds))
display(pd.DataFrame(costs))
print('Framing and guards:', GUARD_BITS*2 + len(PREAMBLE) + HEADER_BITS, 'bit slots')


## 3. Physical data, baseline and observed errors

The development set contains eight earlier packets. The later batch recordings contain 22 packets. Receiver settings were frozen before evaluating the new processing methods on those 22 packets, but the videos and their original baseline results had already been inspected. This is not a blind test on an untouched dataset.

Logs are included under `data/recordings` and `data/experiments`; optional release downloads restore silent copies to those same paths. Study features, metrics and figures are included under `data/analysis`. Each method is evaluated on the same recorded packet when comparing receiver processing.


In [ ]:
from pathlib import Path
import json, hashlib
import numpy as np
import pandas as pd
from IPython.display import display, Image
from optical_link.receiver_study import evaluate, summarize
RX_STUDY = DATA_ROOT / 'analysis/receiver_layers_2026-10-24'
rx_config = json.loads((RX_STUDY / 'frozen_config.json').read_text())
assert hashlib.sha256((RX_STUDY / 'manifest.json').read_bytes()).hexdigest() == rx_config['geometry_manifest_sha256']
assert hashlib.sha256((CODE_ROOT / 'optical_link/receiver_layers.py').read_bytes()).hexdigest() == rx_config['receiver_source_sha256']
alpha, quantile = rx_config['reference_alpha'], rx_config['spatial_quantile']
print(f'Frozen reference gain: {alpha}; spatial quantile: {quantile}; grid: 5×5')
results = {}
def run_layer(method):
    results[method] = evaluate(DATA_ROOT, RX_STUDY, method, alpha=alpha, quantile=quantile)
    display(pd.DataFrame([dict(cohort=c, **summarize([r for r in results[method] if r['cohort']==c])) for c in ['development','earlier_batch_evaluation','guided_batch_evaluation']])[['cohort','packets','successes','channel_errors','payload_outputs','payload_errors','goodput_bps']])

In [ ]:
manifest = json.loads((RX_STUDY / 'manifest.json').read_text())
display(pd.DataFrame([dict(video=v['id'], cohort=v['cohort'], packets=len(v['packets'])) for v in manifest]))
print('Videos:', len(manifest), '| physical packets:', sum(len(v['packets']) for v in manifest))


In [ ]:
run_layer('original')
print('All 30 original packet results reproduced.')

### Physical coding comparison

These are descriptive results from different physical transmissions. Group by cohort and condition: their disturbances were not identical. Do not use the physical table alone to claim a universally best code. Decoder failures remain in packet counts and airtime; payload BER exists only where a payload was returned.


In [ ]:
physical_codes = []
original = pd.DataFrame(results['original'])
for (cohort, condition, codec), group in original.groupby(['cohort','condition','codec']):
    selected = [r for r in results['original'] if (r['cohort'],r['condition'],r['codec']) == (cohort,condition,codec)]
    physical_codes.append(dict(cohort=cohort, condition=condition, codec=codec,
                               **summarize(selected)))
display(pd.DataFrame(physical_codes)[['cohort','condition','codec','packets','successes',
    'channel_ber','payload_outputs','payload_ber_with_output','goodput_bps']])


### Direction and clustering of physical errors

Reuse the physical measurements rather than collecting more recordings. Error direction and consecutive wrong decisions describe the observed channel. A run of wrong bits is not necessarily the duration of a light sweep: transmitted ones can remain correct during positive glare.


In [ ]:
physical = json.loads((SIM_STUDY / 'physical_noise.json').read_text())
assert digest(DATA_ROOT / physical['source']) == physical['source_sha256']
display(pd.DataFrame(physical['cohorts']))
display(Image(filename=str(SIM_STUDY / 'physical_noise.png')))
print(physical['interpretation'])


## 4. Extension one — improve the receiver

First control for patch area, then isolate illumination compensation and spatial robustness, and finally combine them. Timing and coding remain fixed. Expected message bits enter only the evaluator, after independent decoding.

Implementation: [receiver_layers.py](code/optical_link/receiver_layers.py). The original receiver and frozen source hashes are preserved.


### 4.1. Control for the sampled patch area
Green outlines show the original ROI, cyan the common study patch and orange the reference regions. Comparing only a small ROI against a larger one would mix geometry changes with algorithm changes.

In [ ]:
display(Image(filename=str(RX_STUDY / 'features/development_geometry.jpg')))
display(Image(filename=str(RX_STUDY / 'features/evaluation_geometry.jpg')))
run_layer('patch_mean')

### 4.2. Illumination compensation
Estimate the lighting change from four non-flashing reference regions, relative to their preamble brightness. Subtract the median reference change from the patch signal. This can remove a shared brightness shift; it cannot assume local glare affects every region equally.

In [ ]:
run_layer('reference')

### 4.3. Spatial robustness
Split the patch into 25 tiles. Normalize each tile using its public-preamble dark/bright levels, then take the lower quartile of their brightness. This reduces positive glare outliers. Dark obstructions can create the opposite problem.

In [ ]:
run_layer('spatial')

### 4.4. Combine the two components
Subtract the reference change from each tile, calibrate from the public preamble and then aggregate spatially. Timing and coding stay fixed. No expected message or transmitted payload bits are passed to this decoder.

In [ ]:
run_layer('combined')
print('Remaining channel errors handled by the existing error-correcting codes:')
display(pd.DataFrame([r for r in results['combined'] if r['body_channel_bit_errors']])[['trial_id','codec','body_channel_bit_errors','payload_bit_errors','crc_ok']])

### 4.5. Compare all layers
Useful rate = total correctly delivered, CRC-valid message bits / total packet airtime, including framing and redundancy. Packet failures remain in that denominator. Channel BER and message BER use different bit populations.

In [ ]:
comparison = pd.DataFrame([dict(method=m, **summarize(rows)) for m,rows in results.items()])
display(comparison[['method','packets','successes','channel_errors','channel_bits','payload_outputs','payload_errors','goodput_bps']])
display(Image(filename=str(RX_STUDY / 'results/recovery_by_layer.png')))
display(Image(filename=str(RX_STUDY / 'results/errors_goodput.png')))

In [ ]:
display(Image(filename=str(RX_STUDY / 'results/paired_packets.png')))
display(Image(filename=str(RX_STUDY / 'results/example_layer_signals.png')))

### 4.6. Audit the parameter selection
Candidate settings are evaluated only on the eight development packets. The combined receiver uses the independently chosen component settings. This cell displays the retained decision record; it does not tune parameters again.

In [ ]:
candidates = json.loads((RX_STUDY / 'development_candidates.json').read_text())
display(pd.DataFrame([dict(method=c['method'], gain=c['alpha'], quantile=c['quantile'], **c['summary']) for c in candidates])[['method','gain','quantile','successes','clean_failures','channel_outputs','channel_errors']])
print(rx_config['selection_rule'])
print('Frozen:', rx_config['frozen_utc'])

### 4.7. What the physical receiver result establishes

- The common-patch control separates geometry from processing. The two isolated components show what each contributes.
- The shared clock is acquired from the original ROI; lost-preamble recovery is not tested by these new layers.
- The 30/30 result applies to this retained dataset. Eight packets influenced parameter choice; the later 22 were evaluated with frozen settings after their original baseline results had been inspected.
- Keep failures, extra sweeps, manual geometry and unequal interference conditions visible. Do not treat multiple packets from one video or neighbouring bits as independent reliability trials.
- Uniform saturation or complete obstruction can remove information. The tests verify that the implementation does not invent a recovery in that case.

The following sections provide the separate controlled coding study. These physical results remain the hardware evidence.


## 5. Controlled synthetic comparison of the codes

The physical sweeps differed. We now isolate coding effects using declared noise models, shared random payloads and shared noise sequences. The fixed simulation uses nine-byte random messages, 2,000 trials per condition and a 200 ms bit period. It assumes correct synchronization and headers while charging all packet overhead to airtime.

Implementation: [coding_experiments.py](code/optical_link/coding_experiments.py). This is an explanatory simulation, not a fitted camera model.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
from IPython.display import display, Image
from optical_link.coding_experiments import (
    DEFAULT_CONFIG, load_verified, make_packet, interleave, deinterleave,
    measure_trial, digest, run_study)
ROOT = CODE_ROOT
STUDY = SIM_STUDY
trials, summary, checks = load_verified(ROOT, STUDY)
config = json.loads((STUDY / 'config.json').read_text())
display(pd.Series(checks, name='Verification'))
print('Synthetic results only. Headers and synchronization are assumed correct.')


### 5.1. Define the noise models

1. **Independent flips:** each transmitted body bit is flipped with probability p.
2. **Correlated flips:** a two-state process flips every bit while in its bad state. Its stationary bad-state fraction is p and its mean bad run is eight bit slots.
3. **Positive glare bursts:** the same type of burst process forces the received bit to one. Its parameter is bad-state occupancy ρ, not BER: already-bright ones are unchanged.

For the burst processes, P(bad→good)=1/L and P(good→bad)=ρ/[(1−ρ)L]. Each trial begins in the stationary distribution. These are explanatory stress tests, not a channel model fitted or validated against the phone videos.

Within a trial, codes and transmission orders see the same noise process in wire time; longer codewords consume a longer prefix. Across trials, payloads and channel draws are independent. The shared seeds enable paired comparisons.


In [ ]:
display(Image(filename=str(STUDY / 'noise_models.png')))
display(pd.Series({k: config[k] for k in ['seed','trials_per_condition','payload_bytes','rates','mean_burst_slots','length_sweep_slots']}))

### 5.2. Compare the existing codes

Success requires **both** an exact payload and a valid CRC. Undetected wrong payloads are recorded separately and never counted as useful delivery. Decoder rejections remain packet failures; their missing payload BER is not replaced with zero.

Shading shows 95% Wilson intervals over independent simulated packet trials. Useful rate includes the failed packets' airtime. It excludes camera-transfer/operator delays and is not a measured wall-clock hardware rate.


In [ ]:
baseline = summary[(summary.depth == 1) & (summary.sweep == 'rate')]
display(baseline[baseline.rate == 0.05][['model','codec','trials','channel_ber','success_rate',
    'success_ci_low','success_ci_high','payload_outputs','decoder_rejections','goodput_bps']])
display(Image(filename=str(STUDY / 'coding_comparison.png')))


### 5.3. Check against analytical predictions

Independent-noise recovery can also be calculated from binomial probabilities: zero errors for uncoded transmission; a correct majority in each repetition group; at most one error per Hamming word; and at most two erroneous bytes for RS. The saved audit checks these predictions against both transmission orders without fitting the theory to the data.


In [ ]:
analytic = pd.DataFrame(json.loads((STUDY / 'iid_analytic_check.json').read_text()))
display(analytic[(analytic.p == 0.05) & (analytic.depth == 1)])


## 6. Extension two — interleaving

Write the encoded body across eight rows and read it down columns. Omit empty cells in the last row, so there is **no transmitted padding**. After the noisy channel, restore the original order before decoding. The codec and CRC are unchanged; the interleaver does not itself correct any bits.

This is a standalone synthetic extension. It has not been added to the physical transmitter/header, and interleaving old received movies would not create a valid physical experiment.


In [ ]:
packet = make_packet('MANUNITED', 'hamming74', 23)
wire = interleave(packet.body_bits, depth=8)
assert np.array_equal(deinterleave(wire, depth=8), packet.body_bits)
print('Original encoded bits:', len(packet.body_bits), '| transmitted bits:', len(wire))
burst = np.arange(len(wire)) < 8
example = [dict(depth=d, **measure_trial(packet, burst, 'burst_flip', d)) for d in [1,8]]
display(pd.DataFrame(example)[['depth','channel_errors','payload_errors','success','airtime_seconds']])
display(Image(filename=str(STUDY / 'interleaving_example.png')))


### 6.1. Measure both benefits and harms

Depth eight was fixed before the main run. Compare the same payload and same time-domain disturbance with and without interleaving. The paired table retains both rescues and harms. Under independent noise, a permutation should not systematically improve the code's probability of success; finite simulations still vary.

Repetition and Hamming work on short bit groups. Reed–Solomon works on bytes across an entire packet: scattering a short burst across many bytes can make it worse. This experiment tests a bit interleaver, not interleaving multiple RS codewords.


In [ ]:
paired = pd.DataFrame(json.loads((STUDY / 'paired_outcomes.json').read_text()))
cases = pd.DataFrame(json.loads((STUDY / 'scenarios.json').read_text()))
paired = paired.merge(cases, on='scenario_id')
display(paired[(paired.rate == 0.05) & (paired.mean_burst_slots == 8)][[
    'model','codec','pairs','interleaving_rescues','interleaving_harms','success_change_percentage_points']])
display(Image(filename=str(STUDY / 'interleaving_comparison.png')))


### 6.2. Burst duration, buffering and undetected errors

Keep bad-state occupancy at 5% and vary mean burst length from 2 to 16 slots. Longer bursts also imply fewer burst starts at the same occupancy: this is not an experiment holding burst count fixed.

Interleaving adds no channel bits here. This implementation buffers the entire encoded body, so the table shows its batch-buffering span. That span is not extra on-air time or a measured latency penalty: the current packet pipeline already waits for a complete packet. The permutation is agreed in advance; a physical adaptive scheme would need signalling and integration.


In [ ]:
display(Image(filename=str(STUDY / 'burst_duration.png')))
display(summary[(summary.model == 'iid_flip') & (summary.rate == 0)][[
    'codec','depth','airtime_per_packet_seconds','offered_payload_bps','buffer_bits','buffer_span_seconds']])
print('CRC-valid wrong-payload events across all synthetic runs:', int(trials.undetected_wrong_payload.sum()))
display(pd.DataFrame(json.loads((STUDY / 'crc_collision_cases.json').read_text())))
print('The live receiver would accept a CRC-valid wrong message. Only the evaluator knows the sent message and can identify this error.')


## 7. Conclusions and limits

- Separate three claims: measured physical coding behaviour; paired receiver improvement on the same videos; controlled synthetic coding/interleaving behaviour.
- Use the noise mechanism to explain results, including cases where more coding or interleaving performs worse.
- Report both success and useful rate, not just residual BER among decoder outputs.
- The simulated confidence intervals quantify Monte Carlo uncertainty under the declared model. They do not quantify reliability of the real camera link.
- The physical receiver's 30/30 outcome is dataset-specific; eight packets influenced parameter selection and the later recordings had already been inspected.
- Fixed nine-byte random messages isolate the main coding comparison. Different payload lengths, camera rates and header failures are outside this simulation's scope.

The final submission still needs the group's 4–6-page report, presentation/live demonstration and contribution forms. Review and understand the code and its assumptions before presenting it.


### Evidence to use in the report

| Claim | Supporting evidence | Limit |
|---|---|---|
| A real optical link carries binary messages and experiences physical interference | Original videos, optical IDs, transmission logs and baseline measurements | Selected recordings from one setup |
| Processing improves recovery on the retained videos | Same-packet comparison: baseline, area control, compensation, spatial processing, combined | Manual geometry; settings chosen on eight packets; later videos previously inspected |
| Coding performance depends on error structure and transmission cost | Controlled independent, correlated and positive-glare simulations; analytical checks | Declared synthetic model and one payload length |
| Interleaving helps some coding units and harms others | Paired interleaving experiments and burst-duration sensitivity | Bit interleaver within one packet; no physical interleaver transmission |

The next assessed deliverables are the **4–6-page report**, presentation with a brief live demonstration, and individual contribution forms. A strong report must justify the design and explain limitations. The separate Moodle marking rubric still needs checking before any mark-band judgment.

Detailed analysis reports: [physical receiver](data/analysis/receiver_layers_2026-10-24/report.html) · [coding and interleaving](data/analysis/coding_experiments_2026-10-24/report.html).


## 8. Optional full simulation rerun

Leave the switch false for the normal walkthrough. Set it true to generate all 480,000 decodes again in a fresh local folder. The existing study is preserved. The runner does not operate hardware or publish anything.


In [ ]:
RUN_FULL_EXPERIMENT = False
if RUN_FULL_EXPERIMENT:
    from datetime import datetime, timezone
    rerun = DATA_ROOT / 'analysis' / ('coding_rerun_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ'))
    run_study(ROOT, rerun, config)
    print('New local results:', rerun)
else:
    print('Using the verified saved run; full simulation rerun is disabled.')


## 9. Supporting code and experiment history

- `code/optical_link/`: protocol, serial control, camera decoding and experiment implementations.
- `code/firmware/` and `code/vendor/`: transmitter firmware and credited display driver.
- `code/tests/`: software verification; see the README for the command.
- `data/experiments/` and `data/results/`: retained experiment records, including earlier attempts. Only the frozen receiver manifest selects the 30 packets used in the reported physical comparison.
- `data/recordings_manifest.json`: optional silent-video downloads, exact restore paths and SHA-256 hashes.
- `report.tex`: the working IEEE report draft, with embedded figures and references.

Original recording notebooks, device restoration files and local environment files are not publication material. Run All does not transmit to the hardware.
